In [ ]:
import torch
import torch.nn as nn
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import cv2

In [ ]:
NUMBER_OF_PICTURES = 1818

In [ ]:
import os
from tqdm import tqdm 
def img_to_array(img):
    return np.asarray(img)
SIZE=200
clean_data=[]
path = 'dataset2.0'
files=os.listdir(path)
for i in tqdm(files):
    img=cv2.imread(path+'/'+i)  #Change 0 to 1 for color images
    img=cv2.resize(img,(SIZE, SIZE))
    img=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    clean_data.append(img_to_array(img))


In [ ]:
clean_data=clean_data[:NUMBER_OF_PICTURES]
plt.imshow(clean_data[0],cmap='binary')
new_clean=[]
for i in clean_data:
    i=i/255
    new_clean.append(i)

In [ ]:
for i in new_clean:
    x=i
    x[x<0.5]=0

In [ ]:
plt.imshow(new_clean[0],cmap='binary')

In [ ]:
len(new_clean)

In [ ]:
arr_ind =  np.arange(NUMBER_OF_PICTURES)
mean_row=[]
for i in arr_ind:
    mean_row.append(np.mean(new_clean[i][120]))
mean_array=np.array(mean_row)

In [ ]:
plt.scatter(arr_ind,mean_array)
plt.plot(arr_ind,mean_array)
plt.show()

In [ ]:
for i in range(len(new_clean)):
    new_clean[i]=new_clean[i][:120]

In [ ]:
df=pd.read_excel('dataset_2.0.xlsx', engine='openpyxl').head(NUMBER_OF_PICTURES)

In [ ]:
y=df['PM2.5']
y=np.array(y)
print(type(y))

In [ ]:
new_clean[0].shape

In [ ]:
X=new_clean
X=np.array(X)

print(type(X))


In [ ]:

plt.imshow(new_clean[0],cmap='binary')

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size = 0.10, random_state = 42,shuffle=True)

In [ ]:
x_train = x_train.reshape(-1,120,200, 3)

x_test = x_test.reshape(-1,120,200, 3)

In [ ]:

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
import math
import time
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Keras defaults that differ from PyTorch defaults
KERAS_BN = dict(eps=1e-3, momentum=0.01)  # BatchNormalization(epsilon=1e-3, momentum=0.99)

def keras_init(m):
    """Initialise like Keras: glorot_uniform kernels, zero biases."""
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class KerasAdam(torch.optim.Optimizer):
    """Adam with the exact Keras update rule (epsilon=1e-7, added before bias correction,
    unlike torch.optim.Adam which adds it after)."""
    def __init__(self, params, lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=1e-7):
        super().__init__(params, dict(lr=lr, beta_1=beta_1, beta_2=beta_2, epsilon=epsilon))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if not state:
                    state['step'] = 0
                    state['m'] = torch.zeros_like(p)
                    state['v'] = torch.zeros_like(p)
                state['step'] += 1
                t = state['step']
                alpha = group['lr'] * math.sqrt(1 - group['beta_2'] ** t) / (1 - group['beta_1'] ** t)
                state['m'].lerp_(p.grad, 1 - group['beta_1'])
                state['v'].lerp_(p.grad.square(), 1 - group['beta_2'])
                p.addcdiv_(state['m'], state['v'].sqrt().add_(group['epsilon']), value=-alpha)

metrics_names = ['loss', 'mean_squared_error', 'root_mean_squared_error']

def to_tensor(x):
    """(N, H, W, C) numpy images -> (N, C, H, W) float32 tensor."""
    return torch.from_numpy(np.ascontiguousarray(np.asarray(x, dtype=np.float32).transpose(0, 3, 1, 2)))

def to_targets(y):
    return torch.from_numpy(np.asarray(y, dtype=np.float32).reshape(-1, 1))

def run_epoch(model, loss_fn, x, y, batch_size, optimizer=None, shuffle=False):
    """One pass over (x, y); trains when an optimizer is given. Returns [loss, mse, rmse]."""
    n = len(x)
    order = torch.randperm(n) if shuffle else torch.arange(n)
    sum_loss = sum_se = 0.0
    model.train(optimizer is not None)
    with torch.set_grad_enabled(optimizer is not None):
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            xb, yb = x[idx].to(device), y[idx].to(device)
            out = model(xb)
            loss = loss_fn(out, yb)
            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            sum_loss += loss.item() * len(idx)
            sum_se += torch.sum((out.detach() - yb) ** 2).item()
    return [sum_loss / n, sum_se / n, math.sqrt(sum_se / n)]

class History:
    def __init__(self):
        self.history = {}

def fit(model, optimizer, loss_fn, x, y, batch_size=32, epochs=1, verbose=1, validation_split=0.0, shuffle=True):
    """Equivalent of keras Model.fit: validation_split holds out the last fraction of x/y before shuffling."""
    x, y = to_tensor(x), to_targets(y)
    x_val = y_val = None
    if validation_split:
        split_at = int(math.floor(len(x) * (1.0 - validation_split)))
        x, x_val = x[:split_at], x[split_at:]
        y, y_val = y[:split_at], y[split_at:]
    steps = math.ceil(len(x) / batch_size)
    hist = History()
    for epoch in range(epochs):
        epoch_start = time.time()
        logs = dict(zip(metrics_names, run_epoch(model, loss_fn, x, y, batch_size, optimizer, shuffle)))
        if x_val is not None:
            val_scores = run_epoch(model, loss_fn, x_val, y_val, batch_size)
            logs.update({'val_' + k: v for k, v in zip(metrics_names, val_scores)})
        for k, v in logs.items():
            hist.history.setdefault(k, []).append(v)
        if verbose:
            print(f'Epoch {epoch + 1}/{epochs}')
            print(f'{steps}/{steps} - {time.time() - epoch_start:.0f}s - '
                  + ' - '.join(f'{k}: {v:.4f}' for k, v in logs.items()))
    return hist

def evaluate(model, loss_fn, x, y, batch_size=32, verbose=1):
    """Equivalent of keras Model.evaluate. Returns [loss, mse, rmse]."""
    scores = run_epoch(model, loss_fn, to_tensor(x), to_targets(y), batch_size)
    if verbose:
        print(f'{math.ceil(len(x) / batch_size)}/{math.ceil(len(x) / batch_size)} - '
              + ' - '.join(f'{k}: {v:.4f}' for k, v in zip(metrics_names, scores)))
    return scores

def predict(model, x, batch_size=32):
    """Equivalent of keras Model.predict. Returns a (N, 1) numpy array."""
    x = to_tensor(x)
    model.eval()
    outs = []
    with torch.no_grad():
        for start in range(0, len(x), batch_size):
            outs.append(model(x[start:start + batch_size].to(device)).cpu())
    return torch.cat(outs).numpy()

def summary(model, input_size=(3, 120, 200)):
    """Keras-style model.summary(): output shape and parameter count of every layer."""
    rows, hooks = [], []
    for name, module in model.named_modules():
        if len(list(module.children())) == 0:
            hooks.append(module.register_forward_hook(
                lambda m, i, o, name=name: rows.append(
                    (f'{name} ({m.__class__.__name__})', str((None,) + tuple(o.shape[1:])),
                     sum(p.numel() for p in m.parameters())))))
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, *input_size, device=next(model.parameters()).device))
    model.train(was_training)
    for h in hooks:
        h.remove()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Model: "{model.__class__.__name__}"')
    print('_' * 90)
    print(f'{"Layer (type)":<45}{"Output Shape":<30}{"Param #":>15}')
    print('=' * 90)
    for layer, shape, n in rows:
        print(f'{layer:<45}{shape:<30}{n:>15,}')
    print('=' * 90)
    print(f'Total params: {total:,}')
    print(f'Trainable params: {trainable:,}')
    print(f'Non-trainable params: {total - trainable:,}')
    print('_' * 90)

class TransferModel(nn.Module):
    """Equivalent of Model(inputs=base.input, outputs=Dense(1, activation='linear')(Flatten()(base.output)))."""
    def __init__(self, base, input_size=(3, 120, 200)):
        super().__init__()
        self.base = base
        self.flatten = nn.Flatten()
        was_training = base.training
        base.eval()
        with torch.no_grad():
            n_features = self.flatten(base(torch.zeros(1, *input_size, device=next(base.parameters()).device))).shape[1]
        base.train(was_training)
        self.dense = nn.Linear(n_features, 1)
        keras_init(self.dense)

    def train(self, mode=True):
        super().train(mode)
        # Keras runs BatchNormalization layers with trainable=False in inference mode,
        # so frozen BatchNorm layers keep using their ImageNet moving statistics during fit.
        for m in self.base.modules():
            if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
                m.eval()
        return self

    def forward(self, x):
        return self.dense(self.flatten(self.base(x)))

In [ ]:
from torchvision.models import inception_v3, Inception_V3_Weights


inception_v3_full = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
# include_top=False: convolutional base up to Mixed_7c (Keras 'mixed10'), without AuxLogits, pooling and fc
inception = nn.Sequential(
    inception_v3_full.Conv2d_1a_3x3, inception_v3_full.Conv2d_2a_3x3, inception_v3_full.Conv2d_2b_3x3, inception_v3_full.maxpool1,
    inception_v3_full.Conv2d_3b_1x1, inception_v3_full.Conv2d_4a_3x3, inception_v3_full.maxpool2,
    inception_v3_full.Mixed_5b, inception_v3_full.Mixed_5c, inception_v3_full.Mixed_5d,
    inception_v3_full.Mixed_6a, inception_v3_full.Mixed_6b, inception_v3_full.Mixed_6c, inception_v3_full.Mixed_6d, inception_v3_full.Mixed_6e,
    inception_v3_full.Mixed_7a, inception_v3_full.Mixed_7b, inception_v3_full.Mixed_7c,
).to(device)
# don't train existing weights
for param in inception.parameters():
    param.requires_grad = False

# Flatten + Dense(1, activation='linear') on top of the base
# create a model object
model = TransferModel(inception).to(device)

In [ ]:
summary(model)

In [ ]:
from torch.optim import RMSprop
optimum=KerasAdam([p for p in model.parameters() if p.requires_grad], lr=0.000001)
# optimum=RMSprop([p for p in model.parameters() if p.requires_grad], lr=0.00001, alpha=0.9, eps=1e-07); loss_fn = nn.MSELoss()
loss_fn = nn.L1Loss() # loss='mean_absolute_error'; metrics mean_squared_error and root_mean_squared_error are computed by fit/evaluate

In [ ]:
import time
start = time.process_time()
hist = fit(model, optimum, loss_fn, x_train, y_train,
           batch_size=8, epochs=350, validation_split=0.1 )
print(time.process_time() - start)

In [ ]:
predictions = predict(model, x_test)
pred=[]
for i in predictions:
    print(int(i[0]))
    pred.append(int(i[0]))

for i in y_test:
    print(i)

In [ ]:
diff=[]
for i in range(len(pred)):
    diff.append(abs(y_test[i]-pred[i]))
print(sum(diff) / len(diff))

In [ ]:
for i in diff:
    print(i)

In [ ]:
eva = evaluate(model, loss_fn, x_test, y_test)
print(eva)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
# plot the loss
plt.plot(hist.history['mean_squared_error'], label='Train')
plt.plot(hist.history['val_mean_squared_error'], label='Validation')
# plt.plot(mod.history['accuracy'], label='acc_train')
# plt.plot(mod.history['val_accuracy'], label='acc_val')
plt.legend()
#plt.title('Train Loss and Validation Loss in Proposed Neural Network')
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
# plot the loss
plt.plot(hist.history['loss'], label='Train')
plt.plot(hist.history['val_loss'], label='Validation')
# plt.plot(mod.history['accuracy'], label='acc_train')
# plt.plot(mod.history['val_accuracy'], label='acc_val')
plt.legend()
#plt.title('Train Loss and Validation Loss in Proposed Neural Network')
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
# plot the loss
plt.plot(hist.history['root_mean_squared_error'], label='Train')
plt.plot(hist.history['val_root_mean_squared_error'], label='Validation')
# plt.plot(mod.history['accuracy'], label='acc_train')
# plt.plot(mod.history['val_accuracy'], label='acc_val')
plt.legend()
#plt.title('Train Loss and Validation Loss in Proposed Neural Network')
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
print(len(x_test))

In [ ]:
check_20 = 0 #check how many differences are less or equal 20
check_10 = 0 #check how many differences are less 10
check_20_50 = 0 #check how many differences are 20 to 50
check_50_more = 0 #check how many differences are more than 50
check_0 = 0
total=0
for i in diff:
    if i <= 20:
        if i == 0:
            check_0 = check_0 + 1
            total+=1
        elif i < 10:
            check_10 = check_10 + 1
            total+=1
        check_20 = check_20 + 1
    elif (20 < i or i <=50):
        check_20_50 = check_20_50 + 1
        total+=1
    elif i > 50:
        check_50_more = check_50_more + 1
        total+=1
    else:
        print("Too far")

print(check_0)
print(check_10)
print(check_20)
print(check_20_50)
print(check_50_more)
print('Near:',total)

        


In [ ]:
check_20 = 0 #check how many differences are less or equal 20
check_30 = 0 #check how many differences are less or equal 30
check_40 = 0 #check how many differences are less or equal 40
check_50 = 0 #check how many differences are less or equal 60
check_10 = 0 #check how many differences are less 10
check_50_more = 0 #check how many differences are more than 50
check_0 = 0
total=0
for i in diff:
    if i == 0:
        check_0 = check_0 + 1
        total+=1
    elif i <= 10:
        check_10 = check_10 + 1
        total+=1
    elif i <= 20:
        total+=1
        check_20 = check_20 + 1
    elif i <= 30:
        total+=1
        check_30 = check_30 + 1
    elif i <= 40:
        total+=1
        check_40 = check_40 + 1
    elif i <= 50:
        total+=1
        check_50 = check_50 + 1
    elif i > 50:
        check_50_more = check_50_more + 1
        print("Too far")

print(check_0)
print(check_10)
print(check_20)
print(check_20_50)
print(check_50_more)
print('Near:',total)

        


In [ ]:
near_percentage = (total/len(x_test) )* 100

In [ ]:
print(near_percentage)

In [ ]:
len(x_test)

In [ ]:
predic=[]
obser=[]
for i in range(len(pred)):
    predic.append(pred[i])
    obser.append(y_test[i])

In [ ]:
xpoints = np.array(predic)
ypoints = np.array(obser)

plt.scatter(xpoints, ypoints)
plt.xlabel('Predicted Values')
plt.ylabel('Observed Values')
plt.gcf().set_dpi(300)
plt.show()


In [ ]:
my_rho = np.corrcoef(predic, obser)

print(my_rho)

In [ ]:
y = df['PM2.5']
y = np.array(y)

unique, counts = np.unique(y, return_counts=True)


xpoints = np.array(unique)
ypoints = np.array(counts)

plt.hist(xpoints)
plt.xlabel('PM2.5 Index')
plt.ylabel('Frequency of Images')
plt.gcf().set_dpi(300)
plt.show()

In [ ]:
# Integer MAE, as tf.keras.metrics.mean_absolute_error gave for the integer predictions
losas= torch.abs(torch.tensor(np.array(obser)).to(torch.int64) - torch.tensor(np.array(predic)).to(torch.int64)).sum() // len(predic)

In [ ]:
losas

In [ ]:

from sklearn.metrics import r2_score

r2 = r2_score(obser, predic)

In [ ]:
print(r2)

## ResNet50

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
#from torchvision.models import vgg16, VGG16_Weights

In [ ]:
# include_top=False: drop the final average pooling and fc layers
resnet = nn.Sequential(*list(resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).children())[:-2]).to(device)

In [ ]:
# don't train existing weights
for param in resnet.parameters():
    param.requires_grad = False

In [ ]:
with torch.no_grad():
    x = nn.Flatten()(resnet.eval()(torch.zeros(1, 3, 120, 200, device=device)))

In [ ]:
# Flatten + Dense(1, activation='linear') on top of the base

# create a model object
model = TransferModel(resnet).to(device)

In [ ]:
# view the structure of the model
summary(model)

In [ ]:
from torch.optim import RMSprop
optimum=KerasAdam([p for p in model.parameters() if p.requires_grad], lr=0.000001)
# optimum=RMSprop([p for p in model.parameters() if p.requires_grad], lr=0.00001, alpha=0.9, eps=1e-07); loss_fn = nn.MSELoss()
loss_fn = nn.L1Loss() # loss='mean_absolute_error'; metrics mean_squared_error and root_mean_squared_error are computed by fit/evaluate

In [ ]:
import time
start = time.process_time()
hist = fit(model, optimum, loss_fn, x_train, y_train,
           batch_size=8, epochs=350, validation_split=0.1 )
print(time.process_time() - start)

In [ ]:
predictions = predict(model, x_test)
pred=[]
for i in predictions:
    print(int(i[0]))
    pred.append(int(i[0]))

for i in y_test:
    print(i)

diff=[]
for i in range(len(pred)):
    diff.append(abs(y_test[i]-pred[i]))
print(sum(diff) / len(diff))


for i in diff:
    print(i)

In [ ]:
eva = evaluate(model, loss_fn, x_test, y_test)
print(eva)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
# plot the loss
plt.plot(hist.history['mean_squared_error'], label='Train')
plt.plot(hist.history['val_mean_squared_error'], label='Validation')
# plt.plot(mod.history['accuracy'], label='acc_train')
# plt.plot(mod.history['val_accuracy'], label='acc_val')
plt.legend()
#plt.title('Train Loss and Validation Loss in Proposed Neural Network')
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
# plot the loss
plt.plot(hist.history['loss'], label='Train')
plt.plot(hist.history['val_loss'], label='Validation')
# plt.plot(mod.history['accuracy'], label='acc_train')
# plt.plot(mod.history['val_accuracy'], label='acc_val')
plt.legend()
#plt.title('Train Loss and Validation Loss in Proposed Neural Network')
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
# plot the loss
plt.plot(hist.history['root_mean_squared_error'], label='Train')
plt.plot(hist.history['val_root_mean_squared_error'], label='Validation')
# plt.plot(mod.history['accuracy'], label='acc_train')
# plt.plot(mod.history['val_accuracy'], label='acc_val')
plt.legend()
#plt.title('Train Loss and Validation Loss in Proposed Neural Network')
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
predic=[]
obser=[]
for i in range(len(pred)):
    predic.append(pred[i])
    obser.append(y_test[i])

In [ ]:
xpoints = np.array(predic)
ypoints = np.array(obser)

plt.scatter(xpoints, ypoints)
plt.xlabel('Predicted Values')
plt.ylabel('Observed Values')
plt.gcf().set_dpi(300)
plt.show()


In [ ]:
my_rho = np.corrcoef(predic, obser)

print(my_rho)

In [ ]:
y = df['PM2.5']
y = np.array(y)

unique, counts = np.unique(y, return_counts=True)


xpoints = np.array(unique)
ypoints = np.array(counts)

plt.hist(xpoints)
plt.xlabel('PM2.5 Index')
plt.ylabel('Frequency of Images')
plt.gcf().set_dpi(300)
plt.show()

In [ ]:
# Integer MAE, as tf.keras.metrics.mean_absolute_error gave for the integer predictions
losas= torch.abs(torch.tensor(np.array(obser)).to(torch.int64) - torch.tensor(np.array(predic)).to(torch.int64)).sum() // len(predic)
losas

In [ ]:

from sklearn.metrics import r2_score

r2 = r2_score(obser, predic)
r2

## VGG19

In [ ]:
from torchvision.models import vgg19, VGG19_Weights
# include_top=False: convolutional base only (ends with block5_pool)
vgg19_base = vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features.to(device)
# don't train existing weights
for param in vgg19_base.parameters():
    param.requires_grad = False

# Flatten + Dense(1, activation='linear') on top of the base
# create a model object
model = TransferModel(vgg19_base).to(device)
# view the structure of the model
summary(model)

In [ ]:
from torch.optim import RMSprop
optimum=KerasAdam([p for p in model.parameters() if p.requires_grad], lr=0.000001)
# optimum=RMSprop([p for p in model.parameters() if p.requires_grad], lr=0.00001, alpha=0.9, eps=1e-07); loss_fn = nn.MSELoss()
loss_fn = nn.L1Loss() # loss='mean_absolute_error'; metrics mean_squared_error and root_mean_squared_error are computed by fit/evaluate

In [ ]:
import time
start = time.process_time()
hist = fit(model, optimum, loss_fn, x_train, y_train,
           batch_size=8, epochs=350, validation_split=0.1 )
print(time.process_time() - start)

In [ ]:
predictions = predict(model, x_test)
pred=[]
for i in predictions:
    print(int(i[0]))
    pred.append(int(i[0]))

for i in y_test:
    print(i)

diff=[]
for i in range(len(pred)):
    diff.append(abs(y_test[i]-pred[i]))
print(sum(diff) / len(diff))


for i in diff:
    print(i)

In [ ]:
eva = evaluate(model, loss_fn, x_test, y_test)
print(eva)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
# plot the loss
plt.plot(hist.history['mean_squared_error'], label='Train')
plt.plot(hist.history['val_mean_squared_error'], label='Validation')
# plt.plot(mod.history['accuracy'], label='acc_train')
# plt.plot(mod.history['val_accuracy'], label='acc_val')
plt.legend()
#plt.title('Train Loss and Validation Loss in Proposed Neural Network')
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
# plot the loss
plt.plot(hist.history['loss'], label='Train')
plt.plot(hist.history['val_loss'], label='Validation')
# plt.plot(mod.history['accuracy'], label='acc_train')
# plt.plot(mod.history['val_accuracy'], label='acc_val')
plt.legend()
#plt.title('Train Loss and Validation Loss in Proposed Neural Network')
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
# plot the loss
plt.plot(hist.history['root_mean_squared_error'], label='Train')
plt.plot(hist.history['val_root_mean_squared_error'], label='Validation')
# plt.plot(mod.history['accuracy'], label='acc_train')
# plt.plot(mod.history['val_accuracy'], label='acc_val')
plt.legend()
#plt.title('Train Loss and Validation Loss in Proposed Neural Network')
plt.gcf().set_dpi(300)
plt.show()
plt.savefig('LossVal_loss_nonaugmented')


In [ ]:
predic=[]
obser=[]
for i in range(len(pred)):
    predic.append(pred[i])
    obser.append(y_test[i])

In [ ]:
xpoints = np.array(predic)
ypoints = np.array(obser)

plt.scatter(xpoints, ypoints)
plt.xlabel('Predicted Values')
plt.ylabel('Observed Values')
plt.gcf().set_dpi(300)
plt.show()


In [ ]:
my_rho = np.corrcoef(predic, obser)

print(my_rho)

In [ ]:

from sklearn.metrics import r2_score

r2 = r2_score(obser, predic)
r2

## EfficientNetB2

In [ ]:
from torchvision.models import efficientnet_b7, EfficientNet_B7_Weights


class KerasEfficientNetPreprocessing(nn.Module):
    """Rescaling(1/255) + Normalization layers built into keras.applications.EfficientNetB7.

    The images are already scaled to [0, 1] before reaching the model, so, exactly as in
    the Keras version, they are divided by 255 a second time here.
    """
    def __init__(self):
        super().__init__()
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        return (x / 255.0 - self.mean) / self.std


# include_top=False: preprocessing + convolutional base (ends with top_conv/top_bn/top_activation)
efficientnetb7 = nn.Sequential(
    KerasEfficientNetPreprocessing(),
    efficientnet_b7(weights=EfficientNet_B7_Weights.IMAGENET1K_V1).features,
).to(device)

# don't train existing weights
for param in efficientnetb7.parameters():
    param.requires_grad = False

# Flatten + Dense(1, activation='linear') on top of the base
# create a model object
model = TransferModel(efficientnetb7).to(device)
summary(model)

In [ ]:
from torch.optim import RMSprop
optimum=KerasAdam([p for p in model.parameters() if p.requires_grad], lr=0.000001)
# optimum=RMSprop([p for p in model.parameters() if p.requires_grad], lr=0.00001, alpha=0.9, eps=1e-07); loss_fn = nn.MSELoss()
loss_fn = nn.L1Loss() # loss='mean_absolute_error'; metrics mean_squared_error and root_mean_squared_error are computed by fit/evaluate

In [ ]:
import time
start = time.process_time()
hist = fit(model, optimum, loss_fn, x_train, y_train,
           batch_size=8, epochs=350, validation_split=0.1 )
print(time.process_time() - start)

In [ ]:
predictions = predict(model, x_test)
pred=[]
for i in predictions:
    print(int(i[0]))
    pred.append(int(i[0]))

for i in y_test:
    print(i)

diff=[]
for i in range(len(pred)):
    diff.append(abs(y_test[i]-pred[i]))
print(sum(diff) / len(diff))


for i in diff:
    print(i)

In [ ]:
eva = evaluate(model, loss_fn, x_test, y_test)
print(eva)

In [ ]:
predic=[]
obser=[]
for i in range(len(pred)):
    predic.append(pred[i])
    obser.append(y_test[i])

In [ ]:
xpoints = np.array(predic)
ypoints = np.array(obser)

plt.scatter(xpoints, ypoints)
plt.xlabel('Predicted Values')
plt.ylabel('Observed Values')
plt.gcf().set_dpi(300)
plt.show()


In [ ]:
my_rho = np.corrcoef(predic, obser)

print(my_rho)

In [ ]:
predic=[]
obser=[]
for i in range(len(pred)):
    predic.append(pred[i])
    obser.append(y_test[i])
    
xpoints = np.array(predic)
ypoints = np.array(obser)

plt.scatter(xpoints, ypoints)
plt.xlabel('Predicted Values')
plt.ylabel('Observed Values')
plt.gcf().set_dpi(300)
plt.show()


In [ ]:

from sklearn.metrics import r2_score

r2 = r2_score(obser, predic)
r2

## MobileNetV2

In [ ]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# include_top=False: convolutional base only (ends with Conv_1/Conv_1_bn/out_relu)
mobilenetv2 = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1).features.to(device)
# don't train existing weights
for param in mobilenetv2.parameters():
    param.requires_grad = False

# Flatten + Dense(1, activation='linear') on top of the base

# create a model object
model = TransferModel(mobilenetv2).to(device)

summary(model)

In [ ]:
from torch.optim import RMSprop
optimum=KerasAdam([p for p in model.parameters() if p.requires_grad], lr=0.000001)
# optimum=RMSprop([p for p in model.parameters() if p.requires_grad], lr=0.00001, alpha=0.9, eps=1e-07); loss_fn = nn.MSELoss()
loss_fn = nn.L1Loss() # loss='mean_absolute_error'; metrics mean_squared_error and root_mean_squared_error are computed by fit/evaluate

In [ ]:
import time
start = time.process_time()
hist = fit(model, optimum, loss_fn, x_train, y_train,
           batch_size=8, epochs=350, validation_split=0.1 )
print(time.process_time() - start)

In [ ]:
predictions = predict(model, x_test)
pred=[]
for i in predictions:
    print(int(i[0]))
    pred.append(int(i[0]))

for i in y_test:
    print(i)

diff=[]
for i in range(len(pred)):
    diff.append(abs(y_test[i]-pred[i]))
print(sum(diff) / len(diff))


for i in diff:
    print(i)

In [ ]:
eva = evaluate(model, loss_fn, x_test, y_test)
print(eva)

In [ ]:
predic=[]
obser=[]
for i in range(len(pred)):
    predic.append(pred[i])
    obser.append(y_test[i])

In [ ]:
xpoints = np.array(predic)
ypoints = np.array(obser)

plt.scatter(xpoints, ypoints)
plt.xlabel('Predicted Values')
plt.ylabel('Observed Values')
plt.gcf().set_dpi(300)
plt.show()


In [ ]:
my_rho = np.corrcoef(predic, obser)

print(my_rho)

In [ ]:
predic=[]
obser=[]
for i in range(len(pred)):
    predic.append(pred[i])
    obser.append(y_test[i])
    
xpoints = np.array(predic)
ypoints = np.array(obser)

plt.scatter(xpoints, ypoints)
plt.xlabel('Predicted Values')
plt.ylabel('Observed Values')
plt.gcf().set_dpi(300)
plt.show()


In [ ]:

from sklearn.metrics import r2_score

r2 = r2_score(obser, predic)
r2